# EEdit — dvlm Enhanced Composition (2 images/group subset)

Runs **reference-guided composition** using the `dvlm/` enhanced pipeline:
- Reference token K,V attention injection (identity preservation)
- Domain-adaptive eta — high (0.85) to anchor reference identity against scene prior
- Prompt guidance for natural scene integration (shadows, ground contact, depth)
- Seamless/Poisson composite blending at object boundary

Input data downloaded automatically from Google Drive.  
Results uploaded to Drive immediately after each group finishes.

**Run order:** Clone → Pip install → Restart runtime → all remaining cells in order.

In [ ]:
import subprocess, os, shutil

!nvidia-smi || true

# ── Clone YOUR fork of EEdit (must contain the dvlm/ folder) ──────────────────
GITHUB_USER = "AimeeAyat"
REPO_NAME   = "FIA-EDIT-DVLM"
BRANCH      = "EEdit_baseline"

repo_url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

def run_cmd(cmd):
    try:
        subprocess.run(cmd, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print(f"\nERROR executing {' '.join(cmd)}\nGit Error Output:\n{e.stderr}")
        raise

if os.path.exists("/content/EEdit"):
    if not os.path.exists("/content/EEdit/.git"):
        print("Cleaning existing non-git directory at /content/EEdit before cloning.")
        shutil.rmtree("/content/EEdit")
        run_cmd(["git", "clone", repo_url, "/content/EEdit"])
    else:
        print("Repository already exists. Pulling latest changes.")
        run_cmd(["git", "-C", "/content/EEdit", "pull"])
else:
    print("Cloning repository into /content/EEdit.")
    run_cmd(["git", "clone", repo_url, "/content/EEdit"])

%cd /content/EEdit
!git checkout {BRANCH}
!python -c "import dvlm; print('dvlm package found OK')" || echo 'ERROR: dvlm/ not found — push it to GitHub first'



In [ ]:
# Clean conflicting packages
%pip uninstall -y numpy opencv-python opencv-python-headless || true

# Upgrade pip
%pip install -q --upgrade pip

# Install dependencies
%pip install -q --upgrade --force-reinstall --no-cache-dir \
  numpy==1.26.4 \
  torch==2.5.1 torchvision==0.20.1 xformers==0.0.28.post3 \
  diffusers==0.31.0 transformers==4.46.1 accelerate==1.1.0 \
  sentencepiece==0.2.0 safetensors==0.5.2 huggingface_hub==0.26.2 \
  ftfy==6.3.1 einops==0.8.1 omegaconf==2.3.0 \
  pillow==11.1.0 opencv-python-headless==4.10.0.84 \
  gdown lpips torchmetrics

print("Install complete.")
print("IMPORTANT: Runtime -> Restart Session")

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
%cd /content/EEdit
import numpy, torch, transformers
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
!python -c "import dvlm; print('dvlm OK')"

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from datetime import datetime
from pathlib import Path
import os

auth.authenticate_user()
drive_service = build("drive", "v3")

INPUT_FILE_ID    = "1U7BIJZZinAzraAt_T8jAKX3uPa5c-HQa"  # your composition subset ZIP
OUTPUT_FOLDER_ID = "1R0DNBTzobpIOyYDZuy5YZ2lc599nTu8Q"  # your Drive output folder

_ts = datetime.now().strftime("%Y%m%d_%H%M")
RUN_FOLDER_ID = None

def _get_run_folder():
    global RUN_FOLDER_ID
    if RUN_FOLDER_ID is None:
        RUN_FOLDER_ID = _gdrive_mkdir(drive_service, f"EEdit_dvlm_{_ts}", OUTPUT_FOLDER_ID)
        print(f"Created run folder: EEdit_dvlm_{_ts}")
    return RUN_FOLDER_ID

def _gdrive_mkdir(service, name, parent_id):
    meta = {"name": name, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]}
    return service.files().create(body=meta, fields="id").execute()["id"]

def _gdrive_upload_file(service, local_path, parent_id):
    meta = {"name": os.path.basename(local_path), "parents": [parent_id]}
    media = MediaFileUpload(local_path, resumable=True)
    service.files().create(body=meta, media_body=media, fields="id").execute()

def _upload_tree(service, local_dir, parent_id):
    total = 0
    for item in sorted(Path(local_dir).iterdir()):
        if item.is_dir():
            child_id = _gdrive_mkdir(service, item.name, parent_id)
            total += _upload_tree(service, str(item), child_id)
        else:
            _gdrive_upload_file(service, str(item), parent_id)
            total += 1
    return total

def upload_task_results(task_name, generated_dir, originals_dir=None):
    task_id = _gdrive_mkdir(drive_service, task_name, _get_run_folder())
    total = 0
    if os.path.isdir(generated_dir):
        gen_id = _gdrive_mkdir(drive_service, "generated", task_id)
        n = _upload_tree(drive_service, generated_dir, gen_id)
        print(f"  generated/: {n} files")
        total += n
    if originals_dir and os.path.isdir(originals_dir):
        orig_id = _gdrive_mkdir(drive_service, "originals", task_id)
        n = _upload_tree(drive_service, originals_dir, orig_id)
        print(f"  originals/: {n} files")
        total += n
    print(f"[{task_name}] uploaded {total} files total.")
    return task_id

print("Google Drive authenticated. Upload helpers ready.")

In [ ]:
import os, shutil, zipfile
from pathlib import Path
from googleapiclient.http import MediaIoBaseDownload

COMPOSITION_DIR = "/content/TF-ICON/inputs"
os.makedirs(COMPOSITION_DIR, exist_ok=True)

def gdrive_download(service, file_id, dest_path):
    req = service.files().get_media(fileId=file_id)
    with open(dest_path, "wb") as fh:
        dl = MediaIoBaseDownload(fh, req, chunksize=64*1024*1024)
        done = False
        while not done:
            status, done = dl.next_chunk()
            print(f"  {status.progress()*100:.0f}%", end="\r")
    print(f"  Downloaded: {os.path.basename(dest_path)}")

def extract_zip(zip_path, dest_dir):
    tmp = zip_path + "_tmp"
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(tmp)
    items = [x for x in os.listdir(tmp) if x != "__MACOSX"]
    src = (os.path.join(tmp, items[0])
           if len(items) == 1 and os.path.isdir(os.path.join(tmp, items[0]))
           else tmp)
    for item in os.listdir(src):
        dst = os.path.join(dest_dir, item)
        if os.path.exists(dst):
            shutil.rmtree(dst) if os.path.isdir(dst) else os.remove(dst)
        shutil.move(os.path.join(src, item), dst)
    shutil.rmtree(tmp, ignore_errors=True)

FILE_ID = INPUT_FILE_ID
meta = drive_service.files().get(fileId=FILE_ID, fields="name,size").execute()
fname = meta["name"]
mb    = int(meta.get("size", 0)) / 1024**2
print(f"Downloading {fname}  ({mb:.1f} MB) ...")
local_zip = f"/content/{fname}"
gdrive_download(drive_service, FILE_ID, local_zip)
print(f"Extracting to {COMPOSITION_DIR} ...")
extract_zip(local_zip, COMPOSITION_DIR)
os.remove(local_zip)

n = sum(1 for _ in Path(COMPOSITION_DIR).rglob("*") if _.is_file())
print(f"\nComposition input: {n} files in {COMPOSITION_DIR}")
for p in sorted(Path(COMPOSITION_DIR).iterdir()):
    if p.is_dir():
        count = sum(1 for _ in p.rglob("*") if _.is_file())
        print(f"  {p.name}/  ({count} files)")

## 1. Hugging Face Weights

Accept the FLUX.1-dev access agreement first: https://huggingface.co/black-forest-labs/FLUX.1-dev

Then run this cell with your HF token.

In [ ]:
from huggingface_hub import login, snapshot_download
from getpass import getpass
import os

token = getpass("Hugging Face token: ")
login(token=token)

os.makedirs("/content/EEdit/weights", exist_ok=True)

snapshot_download(
    repo_id="black-forest-labs/FLUX.1-dev",
    local_dir="/content/EEdit/weights",
    allow_patterns=[
        "flux1-dev.safetensors",
        "transformer/config.json",
        "transformer_config.json",
        "model_index.json",
        "scheduler/*",
        "text_encoder/*",
        "text_encoder_2/*",
        "tokenizer/*",
        "tokenizer_2/*",
        "vae/*",
    ],
)
print("Weights downloaded.")

## 2. Composition Setup — 2 images per group

Organises the TF-ICON data into per-group image configs.  
**Only the first 2 samples per group are used** (Real-Cartoon, Real-Painting, Real-Sketch, Real-Real).

In [ ]:
import os, re, json, shutil, numpy as np
from pathlib import Path
from PIL import Image

SUBSET_PER_GROUP = 2

tf_root    = Path("/content/TF-ICON/inputs")
eedit_root = Path("/content/EEdit/input/composition")
cfg_root   = Path("/content/EEdit/configs/composition")
eedit_root.mkdir(parents=True, exist_ok=True)
cfg_root.mkdir(parents=True, exist_ok=True)

groups = {"Real-Cartoon": [], "Real-Painting": [], "Real-Sketch": [], "Real-Real": []}

def pick(files, patterns, exclude=()):
    for pat in patterns:
        for f in files:
            if re.match(pat, f.name.lower()) and not any(e in f.name.lower() for e in exclude):
                return f
    return None

for group_name in ["Real-Cartoon", "Real-Painting", "Real-Sketch", "Real-Real"]:
    domain_path = tf_root / group_name
    if not domain_path.exists(): continue
    collected = 0
    for idx, sample_dir in enumerate(sorted(domain_path.iterdir())):
        if collected >= SUBSET_PER_GROUP: break
        if not sample_dir.is_dir(): continue
        prompt = sample_dir.name
        files  = [p for p in sample_dir.iterdir() if p.is_file()]
        bg         = pick(files, [r"^bg.*\.(jpg|jpeg|png)$"])
        ref_img    = pick(files, [r"^fg.*\.(jpg|jpeg|png)$", r"^dccf.*\.jpg$"], exclude=("_mask",))
        ref_mask   = pick(files, [r"^fg.*_mask.*\.(png|jpg)$"])
        place_mask = pick(files, [r"^mask_bg_fg.*\.(jpg|png)$"])
        if not all([bg, ref_img, place_mask]): continue

        slug    = f"{group_name[:2]}_{idx:04d}_{prompt[:60]}"
        out_dir = eedit_root / group_name / slug
        out_dir.mkdir(parents=True, exist_ok=True)
        for src in [bg, ref_img, place_mask] + ([ref_mask] if ref_mask else []):
            if src: shutil.copy2(src, out_dir / src.name)

        arr = np.array(Image.open(place_mask).convert("L"))
        ys, xs = np.where(arr > 10)
        if len(xs) == 0: continue
        x1, x2 = int(xs.min()), int(xs.max())
        y1, y2 = int(ys.min()), int(ys.max())

        entry = {
            "prompt":      prompt,
            "main_image":  str(out_dir / bg.name).replace("/content/EEdit/", "./"),
            "ref_image":   str(out_dir / ref_img.name).replace("/content/EEdit/", "./"),
            "ref_segment": str(out_dir / (ref_mask.name if ref_mask else ref_img.name)).replace("/content/EEdit/", "./"),
            "x1": x1, "y1": y1, "x2": x2, "y2": y2
        }
        groups[group_name].append(entry)
        collected += 1

for group, imgs in groups.items():
    out_json = cfg_root / f"{group}.json"
    with open(out_json, "w") as f:
        json.dump({"imgs": imgs}, f, indent=2)
    print(f"{group}: {len(imgs)} samples -> {out_json}")

## 3. Run dvlm Enhanced Composition

Uses `dvlm/composition_gen.py` with domain-specific configs from `dvlm/domain_configs/`.

Active improvements:
- **Reference K,V injection** — model attends to actual reference appearance at every attention layer
- **High eta (0.85)** — anchors generation to the pasted reference, prevents FLUX from re-generating the object from text
- **Integration-only prompts** — guides ground contact and shadows without re-styling the object
- **Seamless blending** — soft boundary at the paste edge

Add `--use_tail_cfg` to the command below to enable negative-prompt CFG on the last 5 steps.

In [ ]:
%cd /content/EEdit
import gc, torch, json, os
from datetime import datetime
gc.collect(); torch.cuda.empty_cache()

RUN_TAG     = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = f"EEdit_outputs/dvlm_{RUN_TAG}"
print(f"Output root: {OUTPUT_ROOT}")

GROUP_CFGS = [
    ("Real-Cartoon",  "dvlm/domain_configs/RC_config.json"),
    ("Real-Painting", "dvlm/domain_configs/RP_config.json"),
    ("Real-Sketch",   "dvlm/domain_configs/RS_config.json"),
    ("Real-Real",     "dvlm/domain_configs/RR_config.json"),
]

for group, cfg_file in GROUP_CFGS:
    img_cfg = f"/content/EEdit/configs/composition/{group}.json"
    if not os.path.exists(img_cfg):
        print(f"Skipping {group} -- no config"); continue
    with open(img_cfg) as f:
        n = len(json.load(f)["imgs"])
    if n == 0:
        print(f"Skipping {group} -- 0 images"); continue

    out_dir = f"{OUTPUT_ROOT}/{group}"
    print(f"\n=== {group} ({n} images) -> {out_dir} ===")
    os.makedirs(out_dir, exist_ok=True)
    !python dvlm/composition_gen.py \
      --weights_dir ./weights \
      --config_path ./{cfg_file} \
      --img_config  ./configs/composition/{group}.json \
      --output_dir  ./{out_dir} \
      --use_predefine 1 \
      --no_ref_inject \
      --no_color_harmonize \
      --no_seamless

    print(f"Uploading {group} results...")
    from pathlib import Path
    upload_task_results(f"{group}_{RUN_TAG}", out_dir)

print("\nAll groups done.")
total = sum(len(list(Path(f"{OUTPUT_ROOT}/{g}").glob("*.png")))
            for g, _ in GROUP_CFGS if Path(f"{OUTPUT_ROOT}/{g}").exists())
print(f"Total images generated: {total}")

## 4. Evaluation Metrics

Computes background preservation (MSE / PSNR / SSIM / LPIPS) and CLIP text-image score.

In [ ]:
%pip install -q lpips torchmetrics
print("Metric deps ready.")

In [ ]:
%cd /content/EEdit
import os, json, sys
import numpy as np
import torch
from PIL import Image
from pathlib import Path
import torchvision.transforms as T
from tqdm import tqdm

sys.path.insert(0, "/content/EEdit/img_metrics")
from calculate_ssim  import calculate_ssim
from calculate_psnr  import calculate_psnr
from calculate_lpips import calculate_lpips
from calculate_mse   import calculate_mse
from torchmetrics.multimodal.clip_score import CLIPScore

SIZE   = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

clip_metric = CLIPScore(model_name_or_path="openai/clip-vit-large-patch14").to(DEVICE)
clip_metric.eval()

def load_t(path):
    return T.ToTensor()(Image.open(path).convert("RGB").resize((SIZE, SIZE)))

def load_mask_t(path):
    return T.ToTensor()(Image.open(path).convert("L").resize((SIZE, SIZE)))

def find_mask(cfg_item):
    bg_path = cfg_item["main_image"].lstrip("./")
    for ext in [".jpg", ".jpeg", ".png"]:
        candidate = os.path.join(os.path.dirname(bg_path), f"mask_bg_fg{ext}")
        if os.path.exists(candidate):
            return load_mask_t(candidate), "file"
    x1, y1, x2, y2 = cfg_item["x1"], cfg_item["y1"], cfg_item["x2"], cfg_item["y2"]
    orig_img = Image.open(bg_path)
    ow, oh = orig_img.size
    sx, sy = SIZE / ow, SIZE / oh
    mask = torch.zeros(1, SIZE, SIZE)
    mask[0, int(y1*sy):int(y2*sy), int(x1*sx):int(x2*sx)] = 1.0
    return mask, "bbox"

comp_results = {}
GROUPS = ["Real-Cartoon", "Real-Painting", "Real-Sketch", "Real-Real"]

for group in GROUPS:
    cfg_path = f"configs/composition/{group}.json"
    gen_dir  = Path(f"{OUTPUT_ROOT}/{group}")
    if not gen_dir.is_dir() or not os.path.exists(cfg_path): continue
    with open(cfg_path) as f:
        cfg = json.load(f)

    gen_files = sorted(gen_dir.glob("*.png"))
    if not gen_files:
        print(f"No outputs for {group} — skipping"); continue

    group_results = []
    print(f"\n{group} ({len(gen_files)} images):")
    for i, gf in enumerate(tqdm(gen_files)):
        if i >= len(cfg["imgs"]): break
        item = cfg["imgs"][i]
        bg_path = item["main_image"].lstrip("./")
        if not os.path.exists(bg_path): continue

        gen  = load_t(str(gf))
        orig = load_t(bg_path)
        mask_t, mtype = find_mask(item)
        bg_mask = (mask_t < 0.5).float()
        orig_bg = orig * bg_mask
        gen_bg  = gen  * bg_mask

        with torch.no_grad():
            img_uint8 = (gen.permute(1,2,0).numpy() * 255).clip(0,255).astype("uint8")
            clip_t = clip_metric(
                torch.from_numpy(img_uint8).permute(2,0,1).unsqueeze(0).to(DEVICE),
                [item["prompt"]]
            ).item()

        group_results.append({
            "file":  gf.name,
            "prompt": item["prompt"],
            "mask_source": mtype,
            "mse":   calculate_mse(orig_bg, gen_bg)["value"],
            "psnr":  calculate_psnr(orig_bg, gen_bg)["value"],
            "ssim":  calculate_ssim(orig_bg, gen_bg)["value"],
            "lpips": calculate_lpips(orig_bg, gen_bg, DEVICE)["value"][0],
            "clip":  clip_t,
        })

    n   = len(group_results)
    avg = {k: sum(r[k] for r in group_results)/n for k in ("mse","psnr","ssim","lpips","clip")}
    comp_results[group] = {"individual": group_results, "average": avg, "count": n}
    print(f'{group} ({n}): MSE={avg["mse"]:.4f}  PSNR={avg["psnr"]:.2f}  '
          f'SSIM={avg["ssim"]:.4f}  LPIPS={avg["lpips"]:.4f}  CLIP={avg["clip"]:.2f}')

all_items = [r for g in comp_results.values() for r in g["individual"]]
comp_avg  = {}
if all_items:
    n = len(all_items)
    comp_avg = {k: sum(r[k] for r in all_items)/n
                for k in ("mse","psnr","ssim","lpips","clip")}
    print(f'\nOverall ({n} images): MSE={comp_avg["mse"]:.4f}  '
          f'PSNR={comp_avg["psnr"]:.2f}  SSIM={comp_avg["ssim"]:.4f}  '
          f'LPIPS={comp_avg["lpips"]:.4f}  CLIP={comp_avg["clip"]:.2f}')

metrics_path = f"EEdit_outputs/metrics/dvlm_metrics_{RUN_TAG}.json"
os.makedirs("EEdit_outputs/metrics", exist_ok=True)
with open(metrics_path, "w") as f:
    json.dump({"run_tag": RUN_TAG, "per_group": comp_results, "overall_average": comp_avg}, f, indent=2)
print(f"Saved {metrics_path}")

In [ ]:
import json, os
metrics_path = f"EEdit_outputs/metrics/dvlm_metrics_{RUN_TAG}.json"
comp = json.load(open(metrics_path)) if os.path.exists(metrics_path) else None

print("=" * 65)
print("  EEdit dvlm Enhanced Composition (2 images / group)")
print("=" * 65)

if comp:
    print(f"\n{'Group':<18} {'N':>4}  {'MSE':>8}  {'PSNR':>7}  {'SSIM':>7}  {'LPIPS':>7}  {'CLIP':>7}")
    print("-" * 65)
    for group, data in comp["per_group"].items():
        a = data["average"]
        n = data["count"]
        print(f"{group:<18} {n:>4}  {a['mse']:>8.4f}  {a['psnr']:>7.2f}  {a['ssim']:>7.4f}  {a['lpips']:>7.4f}  {a['clip']:>7.2f}")
    print("-" * 65)
    a = comp["overall_average"]
    n = sum(d["count"] for d in comp["per_group"].values())
    print(f"{'OVERALL':<18} {n:>4}  {a['mse']:>8.4f}  {a['psnr']:>7.2f}  {a['ssim']:>7.4f}  {a['lpips']:>7.4f}  {a['clip']:>7.2f}")

## 5. Upload Metrics to Google Drive

In [ ]:
import os
metrics_dir = "/content/EEdit/EEdit_outputs/metrics"
if os.path.isdir(metrics_dir):
    metrics_id = _gdrive_mkdir(drive_service, "metrics", _get_run_folder())
    n = _upload_tree(drive_service, metrics_dir, metrics_id)
    print(f"Metrics uploaded: {n} files")
else:
    print("No metrics folder found — run the metrics cell first.")
print(f"\nDone! View results at your Drive output folder.")